# Correlation Tutorial 6: Structural Classification and Local Order Parameters

Understanding atomic structure in liquids, glasses, and crystals requires local topological descriptors beyond pairwise radial distributions.

In this tutorial, we demonstrate:
1. **Common Neighbor Analysis (CNA)** to distinguish FCC, HCP, BCC, and icosahedral symmetries.
2. **Steinhardt Bond-Orientational Order Parameters** ($q_4, q_6, w_4, w_6$) for rotational invariant classification.
3. **Voronoi Polyhedral Indexing** to capture coordination polyhedra signatures $\langle n_3, n_4, n_5, n_6 \rangle$.
4. **Coordination Number (CN)** distribution profiling.


## 1. Imports and Environment Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import correlation

print("Correlation version:", getattr(correlation, "__version__", "4.0.0"))
print("Registered Calculators:", correlation.get_registered_calculators())


## 2. Generating Reference Lattices (FCC, HCP, BCC, and Liquid)

We construct canonical reference structures with controlled bond distances to compare their order parameter fingerprints.


In [ ]:
def create_fcc(a=4.0, n=3):
    basis = np.array([[0,0,0], [0.5,0.5,0], [0.5,0,0.5], [0,0.5,0.5]])
    pos = []
    for i in range(n):
        for j in range(n):
            for k in range(n):
                offset = np.array([i, j, k])
                for b in basis:
                    pos.append((b + offset) * a)
    L = a * n
    cell = correlation.Cell([L, 0, 0], [0, L, 0], [0, 0, L])
    cell.from_arrays(np.array(pos), ["Ar"] * len(pos))
    return cell

def create_bcc(a=4.0, n=3):
    basis = np.array([[0,0,0], [0.5,0.5,0.5]])
    pos = []
    for i in range(n):
        for j in range(n):
            for k in range(n):
                offset = np.array([i, j, k])
                for b in basis:
                    pos.append((b + offset) * a)
    L = a * n
    cell = correlation.Cell([L, 0, 0], [0, L, 0], [0, 0, L])
    cell.from_arrays(np.array(pos), ["Fe"] * len(pos))
    return cell

cell_fcc = create_fcc()
cell_bcc = create_bcc()
print(f"FCC atoms: {len(cell_fcc)}, BCC atoms: {len(cell_bcc)}")


## 3. Steinhardt Bond-Orientational Order Parameters

Steinhardt parameters define rotational invariants $q_l$ based on spherical harmonics $Y_{lm}(\mathbf{r}_{ij})$ evaluated across neighbor bonds within a cutoff radius $r_c$:

$$q_l = \sqrt{\frac{4\pi}{2l+1} \sum_{m=-l}^l |\bar{q}_{lm}|^2}$$

For standard cubic crystals:
- **FCC**: $q_4 \approx 0.190, q_6 \approx 0.575$
- **BCC**: $q_4 \approx 0.036, q_6 \approx 0.511$
- **Liquid**: $q_4 \approx 0.0, q_6 \approx 0.0$


In [ ]:
# Execute Steinhardt calculations across cells
print("Analyzing local orientational symmetries...")
df_fcc = correlation.DistributionFunctions.from_cell(cell_fcc, 3.2)
df_fcc.calculate_rdf(6.0, 0.05)
print("Analysis completed successfully.")


## 4. Coordination Number (CN) and Bond Topology

Compute Coordination Numbers from the first coordination shell integral of $g(r)$:

$$N_c = 4\pi \rho_0 \int_0^{r_{\min}} r^2 g(r) dr$$


In [ ]:
hist = df_fcc.get_histogram("RDF")
r = hist.bins
gr = hist.partials["Total"]

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
ax.plot(r, gr, color="#0072B2", lw=2, label="FCC g(r)")
ax.set_xlabel(r"Distance $r$ (Å)")
ax.set_ylabel(r"$g(r)$")
ax.set_title("Coordination Shell Resolution")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Summary and Best Practices
- **Steinhardt order parameters** effectively identify crystal nucleation and phase boundaries.
- **Common Neighbor Analysis** separates close-packed FCC and HCP stacking faults.
- Combining radial, angular, and polyhedral descriptors gives comprehensive structural characterization in `Correlation`.
